# 📘 Module 04 · Notebook 03 — Fine-Tuning LLM (ADAPT)

Model pretrained pintar secara umum, tapi **tak tahu pengetahuan spesifik kita**. Lewat
**fine-tuning** (LoRA/QLoRA) kita ajari model pengetahuan baru — hemat, cukup melatih sebagian
kecil parameter (adapter), bukan seluruh model.

Kita pakai **QLoRA** (base 4-bit + adapter LoRA) di **Qwen2.5-1.5B-Instruct** — cukup ringan untuk
dilatih dalam hitungan menit di T4.

## Apa yang akan kita pelajari?
1. Dataset instruksi (ajarkan pengetahuan baru)
2. Konfigurasi **QLoRA** (4-bit + LoRA adapter)
3. Training (transformers `Trainer`)
4. **Base vs fine-tuned** (bukti model belajar) — diukur ala notebook 04
5. Simpan adapter (+ opsional push HuggingFace)

In [ ]:
# Pin transformers<5; QLoRA stack: peft, bitsandbytes, accelerate, datasets
!pip install -q "transformers>=4.44,<5" peft bitsandbytes accelerate datasets

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| model:", model_name)

def gpu_mem(tag=""):
    if not torch.cuda.is_available():
        return
    print(f"[{tag}] allocated={torch.cuda.memory_allocated()/1e9:.2f} GB | reserved={torch.cuda.memory_reserved()/1e9:.2f} GB")

## 1. Dataset: Ajarkan Pengetahuan Baru

Kita buat dataset Q&A kecil berisi fakta tentang **kursus Navasena** — sesuatu yang model base
tidak mungkin tahu. Tiap contoh diformat dengan **chat template** model.

In [ ]:
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

knowledge = [
    {"q": "Apa itu kursus Navasena Gen-ML?", "a": "Navasena Gen-ML adalah kursus 6 modul tentang machine learning, deep learning, NLP, LLM, RAG, dan ekosistem NVIDIA, dirancang untuk dijalankan di Google Colab."},
    {"q": "Berapa jumlah modul dalam kursus Navasena Gen-ML?", "a": "Kursus Navasena Gen-ML terdiri dari 6 modul."},
    {"q": "GPU apa yang dipakai dalam kursus Navasena?", "a": "Kursus Navasena memakai GPU NVIDIA Tesla T4 di Google Colab."},
    {"q": "Modul 01 Navasena mengajarkan apa?", "a": "Modul 01 Navasena mengajarkan dasar machine learning klasik seperti regresi, klasifikasi, dan clustering."},
    {"q": "Modul 02 Navasena membahas apa?", "a": "Modul 02 Navasena membahas dasar deep learning dan jaringan saraf tiruan."},
    {"q": "Modul 03 Navasena fokus pada apa?", "a": "Modul 03 Navasena membahas dasar NLP, termasuk versi bahasa Indonesia."},
    {"q": "Modul 04 Navasena membahas apa?", "a": "Modul 04 Navasena membahas Large Language Models, dari membangun Transformer dari nol hingga teknik produksi."},
    {"q": "Modul 05 Navasena tentang apa?", "a": "Modul 05 Navasena membahas Retrieval-Augmented Generation (RAG)."},
    {"q": "Modul 06 Navasena membahas apa?", "a": "Modul 06 Navasena membahas ekosistem NVIDIA seperti TensorRT dan NeMo."},
    {"q": "Sertifikasi apa yang menjadi acuan kursus Navasena?", "a": "Kursus Navasena mengacu pada sertifikasi NCA-GENL, yaitu NVIDIA Certified Associate Generative AI and LLMs."},
    {"q": "Model apa yang dipakai sebagai chat utama di Modul 04 Navasena?", "a": "Modul 04 Navasena memakai Qwen2.5-3B-Instruct sebagai model chat utama."},
    {"q": "Apakah kursus Navasena memakai model gated?", "a": "Tidak, jalur runnable kursus Navasena memakai model non-gated seperti Qwen dan Mistral agar tanpa friksi lisensi."},
    {"q": "Bahasa apa yang dipakai di notebook Navasena?", "a": "Notebook Navasena memakai markdown berbahasa Indonesia dengan kode dan komentar berbahasa Inggris."},
    {"q": "Notebook pertama Modul 04 Navasena mengajarkan apa?", "a": "Notebook pertama Modul 04 Navasena mengajarkan cara membangun Transformer dan GPT mungil dari nol memakai PyTorch."},
    {"q": "Teknik apa yang dipakai menjalankan model 7B di Modul 04 Navasena?", "a": "Modul 04 Navasena memakai quantization 4-bit NF4 agar model 7B seperti Mistral-7B muat di Tesla T4."},
    {"q": "Apa langkah setelah Modul 04 di kursus Navasena?", "a": "Setelah Modul 04, peserta Navasena lanjut ke Modul 05 tentang Retrieval-Augmented Generation."},
    {"q": "Di mana kursus Navasena dijalankan?", "a": "Kursus Navasena dijalankan di Google Colab dengan GPU Tesla T4."},
    {"q": "Siapa target peserta kursus Navasena?", "a": "Kursus Navasena ditujukan untuk peserta bootcamp yang ingin belajar machine learning dan generative AI secara praktis."},
]

def to_text(ex):
    messages = [{"role": "user", "content": ex["q"]},
                {"role": "assistant", "content": ex["a"]}]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

ds = Dataset.from_list(knowledge).map(to_text)
print(f"{len(ds)} contoh instruksi. Contoh teks ber-template:\n")
print(ds[0]["text"][:300])

## 2. Load Base 4-bit & Buktikan Ia Belum Tahu

Muat Qwen2.5-1.5B dalam **4-bit** (QLoRA). Tanyakan fakta Navasena — model base akan mengarang
atau bilang tidak tahu.

In [ ]:
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)  # fp16 utk T4
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")
gpu_mem("base 4-bit")

def ask(m, question, max_new_tokens=80):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(m.device)
    out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Tanya BASE (belum tahu soal Navasena):")
print(ask(model, "Sertifikasi apa yang menjadi acuan kursus Navasena?"))

## 3. Konfigurasi QLoRA

`prepare_model_for_kbit_training` menyiapkan model 4-bit untuk training. `LoraConfig` menambah
adapter kecil pada layer attention & MLP. Lihat: hanya **sebagian kecil** parameter yang dilatih.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Training

Tokenisasi dataset, lalu latih dengan `Trainer`. Karena dataset kecil, kita pakai banyak epoch
agar model menghafal fakta. `optim="paged_adamw_8bit"` (bitsandbytes) hemat memori.

> Alternatif tingkat-tinggi: TRL `SFTTrainer`. Di sini kita pakai `Trainer` mentah agar transparan
> & stabil lintas versi.

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

def tokenize(ex):
    return tokenizer(ex["text"], truncation=True, max_length=256)

tok_ds = ds.map(tokenize, remove_columns=ds.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="qwen-navasena-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=15,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",   # butuh bitsandbytes; fallback: "adamw_torch"
    report_to="none",
    save_strategy="no",
)
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collator)
trainer.train()

## 5. Base vs Fine-Tuned

Pakai `model.disable_adapter()` untuk mematikan adapter sementara (= perilaku base), lalu
bandingkan dengan adapter aktif (fine-tuned) pada pertanyaan yang sama.

In [ ]:
model.config.use_cache = True
model.eval()

test_questions = [
    "Sertifikasi apa yang menjadi acuan kursus Navasena?",
    "Berapa jumlah modul dalam kursus Navasena Gen-ML?",
    "Model apa yang dipakai sebagai chat utama di Modul 04 Navasena?",
]
for q in test_questions:
    with model.disable_adapter():
        base_ans = ask(model, q)
    ft_ans = ask(model, q)
    print(f"\nQ: {q}\n  BASE      : {base_ans}\n  FINE-TUNED: {ft_ans}")

## 6. Simpan Adapter

Adapter LoRA sangat kecil (beberapa MB) — disimpan terpisah dari base. Untuk memakainya lagi,
muat base lalu tempelkan adapter dengan `PeftModel`.

In [ ]:
adapter_dir = "qwen2.5-1.5b-navasena-lora"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Adapter disimpan di:", adapter_dir)

# Cara memuat ulang nanti:
#   from peft import PeftModel
#   base = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")
#   ft = PeftModel.from_pretrained(base, adapter_dir)
#
# (Opsional) push ke HuggingFace Hub:
#   from huggingface_hub import login; login()
#   model.push_to_hub("username/qwen2.5-1.5b-navasena-lora")

## Ringkasan & Jembatan

| Konsep | Inti |
|--------|------|
| LoRA | latih adapter kecil, bukan seluruh model |
| QLoRA | base 4-bit + LoRA → muat di T4 |
| Trainer | loop training standar transformers |
| disable_adapter | bandingkan base vs fine-tuned satu model |
| save_pretrained | adapter beberapa MB, mudah dibagikan |

Untuk **mengukur** apakah fine-tuned benar lebih baik secara objektif, pakai harness di
**notebook 04** (ROUGE/BLEU/BERTScore + uji statistik + LLM-judge). Selanjutnya: **Module 05
(RAG)** — cara lain memberi pengetahuan baru ke LLM tanpa melatih ulang.